# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")
print(f"License: {metadata['license']}")
print(f"Version: {metadata['version']}")
print(f"Published: {metadata['datePublished']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List the available record sets and their @id
record_sets = dataset.metadata.record_sets
print(f"Available Record Sets ({len(record_sets)}):\n")
for rs in record_sets:
    print(f"- Name: {getattr(rs, 'name', None)} | @id: {rs.id}")

# For each record set, list fields and columns by their @id
for rs in record_sets:
    print(f"\nRecord Set: {getattr(rs, 'name', None)} (@id: {rs.id})")
    print("  Fields:")
    for field in getattr(rs, 'fields', []):
        print(f"   - {getattr(field, 'name', None)} (@id: {field.id}, dataType: {getattr(field, 'data_type', None)})")
    print("  Columns:")
    for col in getattr(rs, 'columns', []):
        print(f"   - {getattr(col, 'name', None)} (@id: {col.id})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Define all the record set @ids from the Croissant schema
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]

# Load each record set into a DataFrame
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns in '{record_set_id}': {df.columns.tolist()}")
        print(df.head())
    else:
        print(f"No records found for {record_set_id}")

# For continued analysis, select the main protein record set (by inspecting the record set ids above)
# Example: if the main record set id is 'cr:ProteinRecordSet', set it here (replace as necessary)
# If in doubt, use the first non-empty record set
main_record_set_id = None
for rid, df in dataframes.items():
    main_record_set_id = rid
    break
if main_record_set_id is not None:
    print(f"\nMain analysis will use record set: {main_record_set_id}")
    print(dataframes[main_record_set_id].head())
else:
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes filtering, normalization, and grouping for key protein abundance or other quantitative fields.

In [ ]:
import numpy as np

# --- CONFIG: identify a numeric field (by its @id or DataFrame column name) ---
# We'll select the first numeric-appearing field as an example.
numeric_field_id = None
df = dataframes[main_record_set_id]
# Try to infer via column names/ids: pick peptide count, molecular weight, or a numeric abundance
# You can manually override here if known
for col in df.columns:
    # Heuristic for common abundance/weight/count fields
    if any(word in col.lower() for word in ["count", "abund", "weight", "mw", "number", "peptide", "coverage", "%"]):
        numeric_field_id = col
        break
if not numeric_field_id:
    # Fallback: use first float/int
    for col in df.columns:
        if np.issubdtype(df[col].dtype, np.number):
            numeric_field_id = col
            break

print(f"Selected numeric field: {numeric_field_id}\n")
#--- Filtering ---#
threshold = None
# Find an appropriate threshold (10 is default)
col_vals = pd.to_numeric(df[numeric_field_id], errors='coerce')
if col_vals.notna().sum() > 0:
    threshold = np.nanquantile(col_vals, 0.5) # median as threshold
else:
    threshold = 10

filtered_df = df[pd.to_numeric(df[numeric_field_id], errors='coerce') > threshold]
filtered_df = filtered_df.copy()
print(f"Filtered records with {numeric_field_id} > {threshold} (median): {len(filtered_df)} records")
print(filtered_df.head())
#--- Normalization ---#
mean_val = pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').mean()
std_val = pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').std()
filtered_df[numeric_field_id + '_normalized'] = (pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') - mean_val) / std_val
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

#--- Grouping/aggregation ---#
# Try to guess a suitable group field (e.g., protein function/class/description/sample/other)
group_field = None
for col in df.columns:
    if any(word in col.lower() for word in ['sample', 'class', 'desc', 'type', 'group']):
        group_field = col
        break
if group_field and group_field in df.columns:
    # Group and aggregate
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame('mean').join(
        filtered_df.groupby(group_field)[numeric_field_id].count().to_frame('count')
    )
    print(f"\nGrouped data by {group_field} (mean & count):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    # Histogram of numeric field
    plt.figure(figsize=(8,4))
    col_vals = pd.to_numeric(df[numeric_field_id], errors='coerce')
    sns.histplot(col_vals.dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()
    # If we performed grouping, plot group means if available
    if 'grouped_df' in locals() and group_field:
        grouped_df = grouped_df.reset_index()
        plt.figure(figsize=(10,4))
        sns.barplot(data=grouped_df, x=group_field, y='mean')
        plt.title(f"Mean {numeric_field_id} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook demonstrated loading, exploring, and processing the Mass Spectrometry Analysis of Extracellular Vesicles dataset via Croissant metadata and the `mlcroissant` Python library.

- **Metadata** is accessible and gives clear information about data content, license, and publication.
- The schema provides several record sets and fields, referenced by their `@id`.
- After loading the main data, quantitative fields (e.g., peptide counts or protein abundance) can be filtered, normalized, and grouped for basic analytics.
- The fields are dynamically selected using their `@id`, ensuring robust and reproducible exploration workflows.
- Further analysis can include deeper domain-specific insights into protein expression and modifications using the same paradigm.

For more detailed data use or reproducibility, consult the [FAIR² Croissant schema documentation](https://mlcommons.org/croissant/) and the dataset's [SenScience repository](https://app.sen.science/frontiers/7154140/).